# ANN (FNN)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("DateFruit_Dataset.csv")
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
X = df.drop(columns = "Class")
y = df["Class"]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
df["Class"].unique()

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)

# FNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader 

In [ ]:
X_train_tensor = torch.tensor(X_train_scale, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scale, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# Build our ANN model

class FNN(nn.Module):
    def __init__(self):
        super(FNN, self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(64, 64),
            nn.ReLU(),

            # o/p layer
            nn.Linear(64, 7),
        )
    def forward(self, x):
        return self.model(x)

In [ ]:
model = FNN()

# loss and optimizer

crietrion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# Train ANN

epochs = 100
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_dataloader:
        optimizer.zero_grad()
    
        outputs = model(xb)
        loss = crietrion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_dataloader)
    print(f"epoch: {epoch+1} / {epochs}, loss = {train_loss}")
    

In [ ]:
# Evaluate.

model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_dataloader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1) # ex for tensor=[2,5,3,7,3,2,5], this max returns 2 values (max_value, idx_of_that_value) and idx is the predicted value, max_value is not of any use so _ means we dont want to store this value 

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # we can directly do y_test.shape to check this but it is a alternate way of finding the actual samples in each batch

print("Accuracy:", correct / total) # We could use evaluation metrics like precision classification report and many more but, this is the simplest way to check the accuracy